# DPointNet tutorial with V1 network

This notebook introduces dpointnet with networks that respond to visual stimuli. The first section uses a 200-um radius L4 cutout driven by LGN inputs in response to drifting-gratings. It demonstrates how you can tune the connection weights to minimize loss functions. We define a few different firing-rate loss functions to see how it affects rasters and firing-rate distributions. The second section runs the full V1 column for 0 and 180 degree drifting gratings, and adds a non-trivial L5 spatial-bias loss function.


## What DPointNet gives us

DPointNet provides a simulator that runs SONATA-formatted point-neuron networks with GLIF3 models. The DPointNet simulator cotains both the module to process input (where FilterNet takes care in conventional BMTK simulations) and the module to process recurrent network (where PointNet takes care). The network receives stimulus-driven LGN spikes and background input, then produces V1 spikes and voltages. The simulations are run on GPUs when available using TensorFlow as backend. The entire simulation is differentiable; we can train selected synaptic weights with losses defined on firing rates, membrane voltage, and weight distributions. In this tutorial the trainable weights are V1-to-V1 recurrent weights and background input weights; LGN stimulus weights stay fixed.


## Setup

Run this notebook with an environment that has a GPU and TensorFlow available. The tutorial folder already contains the V1 network files, Neuropixels firing-rate targets, synchronization data, JSON config files, and the small `run_dpointnet.py` runner script used by the examples.

Set `RUN_SIMULATIONS = False` when you only want to just replot existing results.

Some plotting utilities live in `workshop_dpointnet.py`; the simulator setup itself stays in the config files and runner script.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

import workshop_dpointnet as ws
ws = importlib.reload(ws)

if Path.cwd().name != 'tutorial':
    os.chdir(Path('tutorial').resolve())

RUN_SIMULATIONS = True

# Setting up a function to run the configuration file
def run_config(config_file):
    command = [sys.executable, 'run_dpointnet.py', str(config_file)]
    print('$ ' + ' '.join(command))
    subprocess.run(command, check=True)


print(f'Working directory: {Path.cwd()}')
print('Set RUN_SIMULATIONS = False if you only want to inspect existing outputs.')

## Network files

The full V1 example network is available online to download (link: https://brain-map.org/our-research/computational-modeling/differentiable-biorealistic-v1-model-resources; Follow 'Prebuilt SONATA network download' link). The first section uses a 200-um radius L4 cutout derived from the downloaded full V1 network. The L4 and full V1 networks have the same basic structure (LGN, BKG, recurrent networks), so the DPointNet config structure works for both networks with small changes.

In [ ]:
l4_metadata = ws.create_l4_cutout_network(
    source_dir='GLIF_network/network',
    target_dir='GLIF_network_l4_cutout/network',
    radius=200.0,
    overwrite=False,
)

print(f"L4 cutout ready: {len(l4_metadata)} neurons")

In [ ]:
l4_metadata = ws.load_v1_metadata('GLIF_network_l4_cutout/network')

cutout_summary = pd.DataFrame([{
    'network_dir': 'GLIF_network_l4_cutout/network',
    'n_neurons': len(l4_metadata),
    'n_cell_types': l4_metadata['cell_type'].nunique(),
}])

display(cutout_summary)
display(l4_metadata.head())
display(l4_metadata.groupby('cell_type').size().rename('n_neurons'))

## Training small L4 cutout network

Let's use 0 degree drifting grating stimulus through the LGN network, plus spontaneous background with a constant firing rate at 250 Hz. First, we want to run a simulation without training the network, to see the evoked response of the untrained, plain network. Second, we would train the network to target a single-valued firing rate for all neurons. Third, we will match the overall firing rate distribution to what's observed in the Neuropixels data. Finally, we will match the firing rate distribution for each cell type (Exc, PV, SST, VIP) separately.


### Plain L4 network

First we run the untrained L4 cutout with the 0 degree LGN stimulus and background input. This gives the baseline evoked firing-rate distribution before any weights are trained. The intro runs for 1000 ms, so each neuron's firing rate falls on 1 Hz increments. The raster shows every neuron in the cutout, colored by cell type: excitatory red, PV green, SST blue, and VIP purple.


In [ ]:
if RUN_SIMULATIONS:
    run_config('configs/config.l4.inference.plain.json')
else:
    print('Skipping plain L4 inference. Set RUN_SIMULATIONS = True to run it.')

plain_rates = ws.plot_intro_result(
    'output_intro_l4_plain/spikes.h5',
    network_dir='GLIF_network_l4_cutout/network',
    t_stop=1000.0,
    title='Plain L4 cutout',
    target_rate=5.0,
)

### Training to one target firing rate

The simulation works, but the firing rates of the untrained network is a bit lower than desired 5 Hz. The first training objective is intentionally simple: pull every neuron toward 5 Hz. This is easy to understand, but it is not biologically realistic. After training, compare the raster and histogram with the baseline and notice how a single scalar target can make the firing-rate distribution too narrow.


In [ ]:
if RUN_SIMULATIONS:
    run_config('configs/config.l4.training.uniform_rate.json')
else:
    print('Skipping uniform-rate training. Set RUN_SIMULATIONS = True to run it.')

ws.plot_loss_if_exists('training_callbacks_intro_l4_uniform_rate/losses.csv', 'Uniform target-rate loss')

uniform_rates = ws.plot_intro_result(
    'output_intro_l4_uniform_rate/spikes.h5',
    network_dir='GLIF_network_l4_cutout/network',
    t_stop=1000.0,
    title='After uniform target-rate training',
    target_rate=5.0,
)

### Distribution-matching loss

The next config compares sorted model firing rates with firing rates sampled from the Neuropixels distribution. The custom loss function for this purpose is registered by `tutorial_losses.py`, which is imported by `run_dpointnet.py`. Please see these files to learn how to make and set up custom loss functions.

### Matching the overall Neuropixels distribution

A single target rate ignores the variability between neurons seen in real cortical recordings. Here we compare the model with the Neuropixels firing-rate distribution and train the model to match that overall distribution. This is a better objective than one fixed rate, but it still does not guarantee that each modeled cell type has the right firing-rate distribution.


In [ ]:
neuropixels_rates = ws.load_neuropixels_rates('Neuropixels_data/OSI_DSI_neuropixels_v4.csv.gz')

fig, axes = ws.plot_firing_rate_distribution(
    uniform_rates,
    neuropixels_rates=neuropixels_rates,
    target_rate=5.0,
    title='Uniform target compared with Neuropixels',
)
plt.show()

if RUN_SIMULATIONS:
    run_config('configs/config.l4.training.overall_distribution.json')
else:
    print('Skipping overall distribution training. Set RUN_SIMULATIONS = True to run it.')

ws.plot_loss_if_exists('training_callbacks_intro_l4_overall_distribution/losses.csv', 'Overall distribution loss')

overall_rates = ws.plot_intro_result(
    'output_intro_l4_overall_distribution/spikes.h5',
    network_dir='GLIF_network_l4_cutout/network',
    t_stop=1000.0,
    title='Overall distribution match',
    neuropixels_rates=neuropixels_rates,
)

### Matching distributions by cell type

The same overall distribution can hide mistakes across cell classes. This stage splits the model by cell type and compares each group with the matching Neuropixels target. This is the closest L4 example to the population-aware firing-rate matching used in the full V1 training configs.


In [ ]:
display(ws.summarize_firing_rates(overall_rates, by='cell_type'))
fig, axes = ws.plot_firing_rate_distribution(
    overall_rates,
    neuropixels_rates=neuropixels_rates,
    by='cell_type',
    title='Overall match, split by model cell type',
)
plt.show()

if RUN_SIMULATIONS:
    run_config('configs/config.l4.training.cell_type_distribution.json')
else:
    print('Skipping cell-type distribution training. Set RUN_SIMULATIONS = True to run it.')

ws.plot_loss_if_exists('training_callbacks_intro_l4_cell_type_distribution/losses.csv', 'Cell-type distribution loss')

cell_type_rates = ws.plot_intro_result(
    'output_intro_l4_cell_type_distribution/spikes.h5',
    network_dir='GLIF_network_l4_cutout/network',
    t_stop=1000.0,
    title='Cell-type-aware distribution match',
    neuropixels_rates=neuropixels_rates,
    by='cell_type',
)

### Scaling the same idea back to V1

The compact L4 network is easy to train, but you can also train much larger network with ~67k neurons. The rest of the notebook applies the same pattern to the 400-µm radius full V1 network.


## Inference before training

Now we move to the full V1 network. We first run two 500 ms drifting-grating conditions, 0 and 180 degrees, using fixed grating phase.

Note: Running this cell may take several minuts to create LGN firing rate cache file.

In [ ]:
if RUN_SIMULATIONS:
    run_config('configs/config.v1.inference.dg0.json')
    run_config('configs/config.v1.inference.dg180.json')
else:
    print('Skipping full-network inference. Set RUN_SIMULATIONS = True to run it.')

## Plotting network behavior

The spatial panels summarize firing rate across the x-z plane. Each panel bins neurons into 50 um squares from -400 to 400 um, and showing the average firing over neuorns within the bin. Rows show stimulus direction and layer: 0 degree L4, 0 degree L5, 180 degree L4, and 180 degree L5. Columns show five 100 ms windows across the 500 ms simulation. The color scale is fixed at 0-15 Hz so figures can be compared before and after training.


In [ ]:
ws.plot_spatial_results(
    {
        0: 'output_inference_dg0/spikes_dg0.h5',
        180: 'output_inference_dg180/spikes_dg180.h5',
    },
    network_dir='GLIF_network/network',
    title='Pre-training spatial firing-rate pattern',
)

## Training Step 1: adjust firing-rate distribution

Step 1 trains the larger V1 network with the firing-rate distribution loss, voltage regularization, and recurrent weight regularization. It uses both 0 and 180 degree stimuli but does not yet add a spatial L5 preference. The goal is to move the full network toward a reasonable firing-rate regime before adding the left/right objective.

Running this cell may take ~20 minutes.

In [ ]:
if RUN_SIMULATIONS:
    run_config('configs/config.v1.training.step1.fr.json')
else:
    print('Skipping Step 1 training. Set RUN_SIMULATIONS = True to run it.')

ws.plot_loss_if_exists('training_callbacks_step1_fr/losses.csv', 'Step 1 losses')

### Plot after Step 1

Step 1 writes post-training inference outputs for both drifting-grating directions. Plotting them with the same spatial layout as the baseline makes it easier to see whether firing rates changed for L4 and L5.

Note that there are other layers as well, but we are watching L4 and L5 for simplicity.

In [ ]:
ws.plot_spatial_results(
    {
        0: 'output_inference_step1_dg0/spikes_dg0.h5',
        180: 'output_inference_step1_dg180/spikes_dg180.h5',
    },
    network_dir='GLIF_network/network',
    title='After Step 1 spatial firing-rate pattern',
    show_l5_bias=True,
)

## Training Step 2: L5 left/right bias with z

Step 2 starts from the Step 1 trained weights. Now we want to introduce a non-trivial behavior in the network. We add an L5 spatial objective. For 0 degrees, the config rewards activity on one side of the z axis; for 180 degrees, the sign is reversed. The custom `L5ZAxisBiasLoss` used by this config is registered by `tutorial_losses.py` when the runner starts. Please take a look at the loss function. This loss function biases the firing rate along z-axis, depending on which direction the grating goes. The z-axis is orthogonal to the stimulus direction, so this bias is unexpected without training.

The training may take ~30 minutes to complete.

In [ ]:
if RUN_SIMULATIONS:
    run_config('configs/config.v1.training.step2.l5_z.json')
else:
    print('Skipping Step 2 training. Set RUN_SIMULATIONS = True after Step 1 weights exist.')

ws.plot_loss_if_exists('training_callbacks_step2_l5_z/losses.csv', 'Step 2 losses')

### Plot after Step 2

Step 2 writes its own post-training inference outputs. These panels should be compared with the baseline and Step 1 panels. Indeed, neurons with z > 0 are firing more for 0 degree stimulus and vice versa. The side-bias summary reports the mean L5 firing rate on negative and positive z sides for each stimulus direction.


In [ ]:
ws.plot_spatial_results(
    {
        0: 'output_inference_step2_dg0/spikes_dg0.h5',
        180: 'output_inference_step2_dg180/spikes_dg180.h5',
    },
    network_dir='GLIF_network/network',
    title='After Step 2 spatial firing-rate pattern',
    show_l5_bias=True,
)

## Extra credit

The L5 spatial objective can create the desired left/right contrast while also changing total L5 activity. How would you modify the objective so the network learns a spatial contrast without simply increasing or decreasing total firing? One useful direction is to compare normalized activity between sides, or add a paired penalty that keeps total L5 firing close to its Step 1 value while rewarding the spatial difference.
